# AIHub WelfareCounsel (복지분야 콜상담) — EDA

`/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터`

## 구조 (확인됨) — 발화 단위 JSON
- `{1.Training, 2.Validation}/{라벨링데이터, 원천데이터}/{TL#|TS#|VL#|VS#}_<도메인>/<도메인>/<cat2>/<cat3>/<세션ID>/<발화ID>.{json|wav}`
- **JSON 1개 = 발화 1개** (세션 JSON 아님). JSON 2,049,002 ≈ wav 2,049,025 (차이 23)
- 도메인 3종: 01.대학병원 / 02.광역이동지원센터 / 03.정신건강복지센터
- **오디오 이미 16kHz mono PCM_16** (스마트폰 녹음) → 리샘플 불필요, 복사만!
- `audioPath`는 윈도우 경로(Y:\...)라 무시 → JSON 경로 치환으로 wav 매핑

## 스키마
- `inputText[0].orgtext` = 전사
- `info[0].metadata.{category1,2,3, speaker_type(상담사/고객), speaker_id(SPK###, 전역!), speaker_age, speaker_sex, sptime_all/start/end, rec_device, rec_place}`

## 규모
- train 1,825,456 / **valid 223,546** 발화 ← 벤치마크 대상(매우 큼 → 샘플링 전략 필요할 수 있음)

In [1]:
from pathlib import Path
import re, json, random, numpy as np, pandas as pd
from collections import Counter
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음")

DATA = Path("/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터")

def json_to_wav(jp):
    """라벨 JSON 경로 → 원천 wav 경로 (라벨링→원천, TL→TS, VL→VS)"""
    s = str(jp).replace("라벨링데이터", "원천데이터").replace("/TL", "/TS").replace("/VL", "/VS")
    return Path(s).with_suffix(".wav")

print("DATA 존재:", DATA.is_dir())

DATA 존재: True


## 1. 매니페스트 파서 (발화 단위 JSON)

> valid 223,546개 JSON 전수는 느릴 수 있어 **표본**으로 EDA (`MAX_FILES`). 빌드 때 전수.
> JSON마다 한 발화: orgtext + metadata.

In [2]:
MAX_FILES = 20000   # EDA 표본 (전수는 빌드 스크립트에서)

def build_manifest(label_root, split, max_files=None):
    rows, bad = [], 0
    jsons = sorted(label_root.rglob("*.json"))
    total = len(jsons)
    if max_files and total > max_files:
        jsons = random.Random(0).sample(jsons, max_files)
    for jp in jsons:
        try:
            d  = json.loads(jp.read_text(encoding="utf-8", errors="replace"))
            md = d["info"][0]["metadata"]
            text = d["inputText"][0]["orgtext"]
        except Exception:
            bad += 1; continue
        rows.append({
            "split": split,
            "cat1": md.get("category1"), "cat2": md.get("category2"), "cat3": md.get("category3"),
            "utt": jp.stem, "session": jp.parent.name,
            "speaker": md.get("speaker_id"), "spk_type": md.get("speaker_type"),
            "gender": md.get("speaker_sex"), "age": md.get("speaker_age"),
            "rec_device": md.get("rec_device"), "rec_place": md.get("rec_place"),
            "sptime_all": md.get("sptime_all"),
            "text": text, "wav": str(json_to_wav(jp)),
        })
    df = pd.DataFrame(rows)
    if len(df):
        df["text"] = df["text"].astype("object")
    print(f"[{split}] 전체 JSON {total:,} 중 표본 {len(jsons):,} → 파싱 {len(df):,} (실패 {bad})")
    return df

df_v = build_manifest(DATA/"2.Validation/라벨링데이터", "valid", MAX_FILES)
df_v.head(3)

[valid] 전체 JSON 223,546 중 표본 20,000 → 파싱 20,000 (실패 0)


,split,cat1,cat2,cat3,utt,session,speaker,spk_type,gender,age,rec_device,rec_place,sptime_all,text,wav
0,valid,정신건강복지센터,자살위기개입,친구동료문제,MEN28000485842A071,MEN0004858,SPK246,상담사,여,40대,스마트폰,집,3.547,더 하시고 싶은 이야기는요?,/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터/2.Va...
1,valid,광역이동지원센터,고객대응,차량요청,MOB21200403021B245,MOB2004030,SPK097,고객,남,20대,스마트폰,집,4.647,처벌이나 행정제재를 가할 수 있는 상황은 아니지만,/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터/2.Va...
2,valid,정신건강복지센터,자살위기개입,가정불화,MEN21000626242B067,MEN0006262,SPK162,고객,여,40대,PC,집,6.897,"덕분에 아르바이트도 시작하고, 사람들과 이런저런 얘기를 나누며",/data/ASR/RAW/AIHub_WelfareCounsel/01.데이터/2.Va...


In [3]:
# wav 매핑/결측 점검
chk = df_v.sample(min(300, len(df_v)), random_state=0)
miss = int(chk["wav"].map(lambda p: not Path(p).exists()).sum())
print(f"wav 존재 표본 {len(chk)} 중 없음: {miss}")
print(f"빈 전사: {int((df_v['text'].str.strip()=='').sum())}")
for c in ["speaker", "gender", "age", "spk_type"]:
    print(f"{c} 결측: {int(df_v[c].isna().sum())}")

wav 존재 표본 300 중 없음: 0
빈 전사: 0
speaker 결측: 0
gender 결측: 0
age 결측: 0
spk_type 결측: 0


## 2. 분포 (valid 표본)

In [4]:
d = df_v
d["text_len"] = d["text"].str.len()
for col in ["cat1", "cat2", "spk_type", "gender", "age", "rec_device", "rec_place"]:
    vc = d[col].value_counts(dropna=False)
    print(f"=== {col} ({d[col].nunique()}종) ===")
    print(vc.head(10).to_string()); print()
print("=== cat3 상위 15 ==="); print(d["cat3"].value_counts().head(15).to_string())
print("\n=== 전사 글자 수 ==="); print(d["text_len"].describe().round(1).to_string())

=== cat1 (3종) ===
cat1
광역이동지원센터    7259
정신건강복지센터    6880
대학병원        5861

=== cat2 (7종) ===
cat2
민원        5058
정신건강상담    4366
진료안내      2623
고객대응      2528
자살위기개입    2514
상담        1771
병원이용안내    1140

=== spk_type (2종) ===
spk_type
고객     11429
상담사     8571

=== gender (2종) ===
gender
여    17405
남     2595

=== age (5종) ===
age
40대    5854
30대    5749
50대    4256
60대    2874
20대    1267

=== rec_device (2종) ===
rec_device
스마트폰    18485
PC       1515

=== rec_place (1종) ===
rec_place
집    20000

=== cat3 상위 15 ===
cat3
차량요청       2335
우울증        1820
외래         1558
적용기준       1502
조현병         946
서비스개선요청     872
이용제한        813
예약불만        591
기타서비스불만     568
검사불만        553
외래진료불만      541
가정불화        488
이성문제        442
외로움고독       425
행위중독        418

=== 전사 글자 수 ===
count    20000.0
mean        20.5
std          9.0
min          2.0
25%         14.0
50%         19.0
75%         25.0
max         95.0


## 3. 전사 컨벤션

> orgtext가 깨끗한 문장인지, KsponSpeech식 태그·이중전사가 있는지

In [5]:
txt = df_v["text"].fillna("")
print("특수문자 전수 (상위 25):")
print(txt.str.findall(r"[^가-힣a-zA-Z0-9\s]").explode().value_counts().head(25).to_string())
print(f"\n영문 포함: {txt.str.contains(r'[A-Za-z]').mean()*100:.2f}%  /  숫자 포함: {txt.str.contains(r'[0-9]').mean()*100:.2f}%")
print("\nKsponSpeech식 주석:")
for tag in ["b/", "n/", "l/", "o/", "u/", ")/(", ")(", "@", "(())"]:
    c = int(txt.str.contains(re.escape(tag)).sum())
    print(f"  '{tag}': {c:,}건")
print("\n전사 샘플 8개:")
for t in txt.sample(min(8, len(txt)), random_state=1):
    print("  •", t[:85])

특수문자 전수 (상위 25):
text
.    12410
ㅇ     3589
?     3146
,     1377
>        2
'        2
!        1
/        1
\        1
ㆍ        1
ㅓ        1

영문 포함: 0.06%  /  숫자 포함: 2.38%

KsponSpeech식 주석:
  'b/': 0건
  'n/': 0건
  'l/': 0건
  'o/': 0건
  'u/': 0건
  ')/(': 0건
  ')(': 0건
  '@': 0건
  '(())': 0건

전사 샘플 8개:
  • 제 이름을 왜요?
  • 성인이 되고 나가서 만나는 친구들이 잘못된 친구들인가?
  • 안 그러면 뭘 찾고 있었는지 아니면 찾고 있었다는 것 조차 홀라당 까먹어요.
  • 고객님께서 ㅇㅇ씨를 가르치려고 하듯이 그리고
  • 수술 이후에 다른 병원 다니시면서 치료 받으신 적은 없나요?
  • 차가 잡히고 나서 차가
  • 이렇게 완벽하지 않은 상황이라면
  • 동생분은 지금 언니도 있고 딸들도 있지만,


## 3-1. 청취

In [6]:
for r in df_v.sample(3, random_state=1).itertuples():
    print(f"[{r.cat1}/{r.spk_type}/{r.gender}/{r.age}] {r.text[:70]}")
    if sf and Path(r.wav).exists():
        data, sr = sf.read(r.wav)
        display(Audio(data, rate=sr))
    else:
        print("   (오디오 없음)")

[대학병원/고객/여/40대] 제 이름을 왜요?


[정신건강복지센터/고객/여/50대] 성인이 되고 나가서 만나는 친구들이 잘못된 친구들인가?


[정신건강복지센터/고객/남/60대] 안 그러면 뭘 찾고 있었는지 아니면 찾고 있었다는 것 조차 홀라당 까먹어요.


## 4. 오디오 속성 + sptime_all 대조

> 16k 재확정 + metadata.sptime_all이 파일 길이와 일치하면 duration으로 활용 가능(복사만 하면 되니 측정도 빠르지만)

In [7]:
samp = df_v.sample(min(80, len(df_v)), random_state=0)
rows = []
for r in samp.itertuples():
    try:
        i = sf.info(r.wav)
        rows.append((i.samplerate, i.channels, i.subtype,
                     round(i.frames/i.samplerate, 3), float(r.sptime_all)))
    except Exception as e:
        rows.append(("ERR", str(e)[:20], "", None, None))
a = pd.DataFrame(rows, columns=["sr","ch","subtype","dur_file","sptime_all"])
print("sample_rate :", a["sr"].value_counts().to_dict())
print("channels    :", a["ch"].value_counts().to_dict())
print("subtype     :", a["subtype"].value_counts().to_dict())
ok = a[a["sr"] != "ERR"].dropna()
if len(ok):
    diff = (ok["dur_file"] - ok["sptime_all"]).abs()
    print(f"파일 길이 vs sptime_all 차이: 평균 {diff.mean():.3f}s / 최대 {diff.max():.3f}s")
    print("(차이 작으면 sptime_all을 duration으로 사용 가능)")

sample_rate : {16000: 80}
channels    : {1: 80}
subtype     : {'PCM_16': 80}
파일 길이 vs sptime_all 차이: 평균 0.000s / 최대 0.000s
(차이 작으면 sptime_all을 duration으로 사용 가능)


## 5. 화자 분석 — 전역 SPK ID라 train/valid 겹침 점검 가능

In [8]:
print(f"valid 표본 고유 화자: {df_v['speaker'].nunique()}명")
print(df_v.groupby("spk_type")["speaker"].nunique().to_string())
print("\n화자별 발화 수: 평균 %.0f / 중앙 %d / 최대 %d" % (
    df_v["speaker"].value_counts().mean(),
    df_v["speaker"].value_counts().median(),
    df_v["speaker"].value_counts().max()))

# train 표본과 화자 겹침 (Zeroth식 speaker-disjoint 점검)
df_t = build_manifest(DATA/"1.Training/라벨링데이터", "train", 5000)
tr, va = set(df_t["speaker"]), set(df_v["speaker"])
print(f"\ntrain 표본 화자 {len(tr)} / valid 표본 화자 {len(va)} / 겹침 {len(tr & va)}")
print("⚠ 겹침이 많으면 valid는 '새 화자' 평가가 아님 (표본 기준 참고치)")

valid 표본 고유 화자: 181명
spk_type
고객     136
상담사    100

화자별 발화 수: 평균 110 / 중앙 87 / 최대 480
[train] 전체 JSON 1,825,456 중 표본 5,000 → 파싱 5,000 (실패 0)

train 표본 화자 252 / valid 표본 화자 181 / 겹침 175
⚠ 겹침이 많으면 valid는 '새 화자' 평가가 아님 (표본 기준 참고치)


In [9]:
# ============================================================
# 비식별화(PII) 토큰 포함 발화 → 전사 + 음성 직접 청취  (모든 EDA 노트북 공용)
# 파서로 DataFrame을 만든 셀을 먼저 실행한 뒤, 이 셀을 새 셀에 붙여 실행.
# DataFrame 변수(df/df_v/...)와 오디오 경로 컬럼(audio_path/wav/...)을 자동 탐지.
# ============================================================
import re
from pathlib import Path
from IPython.display import Audio, display
try:
    import soundfile as sf
except ImportError:
    sf = None
    print("⚠ soundfile 없음 → conda activate TRAIN-ASR")

import pandas as pd

# ---------- 설정 ----------
N_LISTEN = 5          # 들어볼 발화 수
RANDOM_STATE = 0      # None이면 매번 다른 표본
# 비식별화(PII) 토큰: 음향 토큰(b/ n/ l/ o/ u/)과 구분되는 익명화 전용 패턴
PII_PATTERNS = {
    "@ (이름 마커)":           re.compile(r"@"),
    "ㅇㅇ류 (익명화 2자+)":     re.compile(r"ㅇ{2,}"),
    "name/ (이름 태그)":        re.compile(r"(?:^|\s)name/", re.I),
    "[마스킹]":                re.compile(r"\[[^\]]{0,15}\]"),
    "<마스킹>":                re.compile(r"<[^>]{0,15}>"),
    "*** (별표 2+)":           re.compile(r"\*{2,}"),
    "xxx (엑스 2+)":           re.compile(r"[xX]{2,}"),
    "○○ (공백원 2+)":          re.compile(r"[○◯]{2,}"),
}

# ---------- 1) text DataFrame 자동 탐지 ----------
def _find_text_frame():
    g = globals()
    for name in ["df_v","df_valid","df","df_t","df_train","d","a"]:
        o = g.get(name)
        if isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    for name, o in g.items():
        if not name.startswith("_") and isinstance(o, pd.DataFrame) and "text" in o.columns:
            return name, o
    return None, None

_name, _df = _find_text_frame()
if _df is None:
    raise RuntimeError("text 컬럼 DataFrame 없음 — 파서 셀을 먼저 실행하세요.")

# ---------- 2) 오디오 경로 컬럼 자동 탐지 ----------
AUDIO_COL = next((c for c in ["audio_path","wav","src_wav","audio","path","filepath"]
                  if c in _df.columns), None)
print(f"[대상] DataFrame '{_name}' · {len(_df):,} 발화 · 오디오 컬럼: {AUDIO_COL or '없음(PCM 직접 노트북일 수 있음)'}\n")

# ---------- 3) PII 토큰 집계 ----------
_txt = _df["text"].fillna("").astype("object")
print("=== 비식별화 토큰 집계 (text 원본) ===")
present = []
for label, pat in PII_PATTERNS.items():
    occ = int(_txt.str.count(pat).sum())
    utt = int(_txt.str.contains(pat).sum())
    if occ:
        present.append((label, pat, utt, occ))
        print(f"  {label:20s} 발화 {utt:>6,} · 출현 {occ:>6,} ({utt/len(_df)*100:.3f}%)")
if not present:
    print("  → 이 데이터셋엔 정의된 비식별화 토큰이 없음 (자유발화/낭독 등). 청취 생략.")

# ---------- 4) PII 포함 발화만 필터 → 전사 + 음성 재생 ----------
if present and AUDIO_COL:
    mask = pd.Series(False, index=_df.index)
    for _, pat, _, _ in present:
        mask |= _txt.str.contains(pat)
    hits = _df[mask]
    print(f"\n=== 비식별화 토큰 포함 발화 {len(hits):,}건 중 {min(N_LISTEN,len(hits))}개 청취 ===")
    print("   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)\n")
    sample = hits.sample(min(N_LISTEN, len(hits)), random_state=RANDOM_STATE)
    for r in sample.itertuples():
        text = getattr(r, "text", "")
        ap = getattr(r, AUDIO_COL, None)
        # 어떤 PII 패턴에 걸렸는지 표시
        tags = [lab for lab, pat, _, _ in present if pat.search(text or "")]
        print(f"[{', '.join(tags)}]")
        print(f"  전사: {text[:100]}")
        if sf and ap and Path(str(ap)).exists():
            try:
                data, sr = sf.read(str(ap))
                display(Audio(data, rate=sr))
            except Exception as e:
                print(f"   (재생 실패: {str(e)[:50]})")
        else:
            print(f"   (오디오 경로 없음/미존재: {ap})")
        print()
elif present and not AUDIO_COL:
    print("\n⚠ PII 토큰은 있으나 오디오 경로 컬럼을 못 찾음.")
    print("  이 노트북이 PCM을 직접 읽는 방식이면, 아래처럼 수동 지정:")
    print("  → sample = _df[mask].sample(N_LISTEN); 각 행의 키로 원본 PCM 경로를 구성해 재생")

[대상] DataFrame 'df_v' · 20,000 발화 · 오디오 컬럼: wav

=== 비식별화 토큰 집계 (text 원본) ===
  ㅇㅇ류 (익명화 2자+)        발화  1,137 · 출현  1,365 (5.685%)

=== 비식별화 토큰 포함 발화 1,137건 중 5개 청취 ===
   (전사의 마스킹 부분이 음성에서 실제로 어떻게 발화되는지 직접 확인)

[ㅇㅇ류 (익명화 2자+)]
  전사: ㅇㅇ시 ㅇㅇ동 00지에 위치한 ㅇㅇ병원까지 접수해 드렸습니다.



[ㅇㅇ류 (익명화 2자+)]
  전사: ㅇㅇㅇ입니다.



[ㅇㅇ류 (익명화 2자+)]
  전사: 이름은 ㅇㅇㅇ 고요,



[ㅇㅇ류 (익명화 2자+)]
  전사: ㅇㅇ구 ㅇㅇ동이에요.



[ㅇㅇ류 (익명화 2자+)]
  전사: 지금 변경하시면 아버지께서는 목 디스크 전문의이신 ㅇㅇㅇ 교수님이 이제부터의 주치의이시고 계속 ㅇㅇㅇ 교수님이랑 진료를 보셔야 할거에요.
